# Homework 2: Classification

Machine Learning Zoomcamp 2026
- Fabián Bayas Moscoso

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

## Get Data

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


## Data preparation

Check if the missing values are presented in the features.
If there are missing values:

    For categorical features, replace them with 'NA'
    For numerical features, replace with with 0.0



In [4]:
# Check missing values in each feature
print(df.isnull().sum())

# Categorical features: replace missing values with 'NA'
categorical = list(df.select_dtypes(exclude='number').columns)
df[categorical] = df[categorical].fillna('NA')

# Numerical features: replace missing values with 0.0
numerical = list(df.select_dtypes(include='number').columns)
df[numerical] = df[numerical].fillna(0.0)

# Confirm no missing values remain
print(df.isnull().sum())

df.head()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Question 1

What is the most frequent observation (mode) for the column industry?

In [5]:
# Question 1: most frequent value (mode) of the industry column
df['industry'].mode()[0]

'technology'

## Question 2

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?


    interaction_count and lead_score
    number_of_courses_viewed and lead_score
    number_of_courses_viewed and interaction_count
    annual_income and interaction_count

Only consider the pairs above when answering this question.
Split the data

    Split the data with these exact calls:

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

    Make sure that the target value converted is not in your dataframe.


In [6]:
# Question 2: correlation matrix for the numerical features
corr_matrix = df.select_dtypes(include='number').corr()
corr_matrix

# Correlation coefficient for each candidate pair from the question
candidate_pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in candidate_pairs:
    print(f'{a} and {b}: {corr_matrix.loc[a, b]:.4f}')

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


## Question 3

    Calculate the mutual information score between converted and other
    categorical variables in the dataset. Use the training set only.
    Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?

    industry
    location
    lead_source
    employment_status


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

# Split the data with the exact calls from the homework
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# Mutual information between each categorical variable and converted (training set only)
categorical_variables = ['industry', 'location', 'lead_source', 'employment_status']
for col in categorical_variables:
    score = mutual_info_score(df_train[col], df_train['converted'])
    print(f'{col}: {round(score, 2)}')

industry: 0.0
location: 0.0
lead_source: 0.03
employment_status: 0.02


## Question 4

Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.
To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
Calculate the accuracy on the validation dataset and round it to 2 decimal digits.


In [8]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

categorical_features = ['lead_source', 'industry', 'employment_status', 'location']
numerical_features = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
features = categorical_features + numerical_features

y_train = df_train['converted'].values
y_val = df_val['converted'].values

# One-hot encode the categorical features (numerical features are passed through)
dv = DictVectorizer()
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Accuracy on the validation dataset
y_pred = model.predict(X_val)
accuracy = (y_pred == y_val).mean()
print(f'Validation accuracy: {round(accuracy, 2)}')

Validation accuracy: 0.64


## Question 5

Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

    'lead_source'
    'number_of_courses_viewed'
    'interaction_count'

    Note: The difference doesn't have to be positive.


In [9]:
# Question 5: feature elimination using the same features and parameters as Q4
def train_and_score(feature_list):
    dv = DictVectorizer()
    X_train = dv.fit_transform(df_train[feature_list].to_dict(orient='records'))
    X_val = dv.transform(df_val[feature_list].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return (model.predict(X_val) == y_val).mean()

original_accuracy = train_and_score(features)
print(f'Original accuracy (all features): {original_accuracy}')

# Exclude each feature in turn and compare with the original accuracy
for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    reduced_features = [f for f in features if f != feature]
    accuracy_without = train_and_score(reduced_features)
    difference = original_accuracy - accuracy_without
    print(f'{feature}: accuracy without = {accuracy_without}, difference = {difference}')

Original accuracy (all features): 0.645
lead_source: accuracy without = 0.642, difference = 0.0030000000000000027
number_of_courses_viewed: accuracy without = 0.643, difference = 0.0020000000000000018
interaction_count: accuracy without = 0.601, difference = 0.04400000000000004


## Question 6

    Now let's train a regularized logistic regression.
    Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
    Train models using all the features as in Q4.
    Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these C leads to the best accuracy on the validation set?

    0.000001
    0.00001
    0.0001
    0.001

    Note: If there are multiple options, select the smallest C.


In [10]:
# Question 6: regularized logistic regression with different values of C (all features, as in Q4)
dv = DictVectorizer()
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

scores = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    accuracy = (model.predict(X_val) == y_val).mean()
    scores[C] = round(accuracy, 3)
    print(f'C = {C}: validation accuracy = {scores[C]}')

# Best C; ties resolve to the smallest C because the values are iterated in ascending order
best_C = max(scores, key=scores.get)
print(f'Best C: {best_C}')

C = 1e-06: validation accuracy = 0.598
C = 1e-05: validation accuracy = 0.598
C = 0.0001: validation accuracy = 0.613
C = 0.001: validation accuracy = 0.645
Best C: 0.001
